In [1]:
!nvidia-smi

Wed Nov 18 18:05:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
print("Token loaded ✅")

Token loaded ✅


In [3]:
!pip install -q --upgrade kaggle

In [4]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   85M  113G   1% /


In [5]:
!kaggle datasets download -d awsaf49/brats20-dataset-training-validation -p /content
!ls -lh /content

Dataset URL: https://www.kaggle.com/datasets/awsaf49/brats20-dataset-training-validation
License(s): CC0-1.0
100% 4.16G/4.16G [03:50<00:00, 19.4MB/s]

total 4.2G
-rw-r--r-- 1 root root 4.2G Jul  2  2020 brats20-dataset-training-validation.zip
drwxr-xr-x 1 root root 4.0K Nov 17 17:03 sample_data


In [6]:
!unzip -q /content/brats20-dataset-training-validation.zip "BraTS2020_TrainingData/*" -d /content/brats

In [7]:
import os

root = "/content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
patients = sorted([p for p in os.listdir(root) if p.startswith("BraTS20")])
print("Patients:", len(patients))

for p in patients:
    files = os.listdir(f"{root}/{p}")
    if f"{p}_seg.nii" not in files:
        print("⚠️ Seg missing/peyar vera:", p, files)

Patients: 369
⚠️ Seg missing/peyar vera: BraTS20_Training_355 ['BraTS20_Training_355_flair.nii', 'W39_1998.09.19_Segm.nii', 'BraTS20_Training_355_t2.nii', 'BraTS20_Training_355_t1ce.nii', 'BraTS20_Training_355_t1.nii']


In [8]:
old = f"{root}/BraTS20_Training_355/W39_1998.09.19_Segm.nii"
new = f"{root}/BraTS20_Training_355/BraTS20_Training_355_seg.nii"

if os.path.exists(old):
    os.rename(old, new)
    print("Renamed ✅")

Renamed ✅


In [9]:
import os

root = "/content/brats/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
patients = sorted([p for p in os.listdir(root) if p.startswith("BraTS20")])
print("Patients:", len(patients))

for p in patients:
    files = os.listdir(f"{root}/{p}")
    if f"{p}_seg.nii" not in files:
        print("⚠️ Seg missing/peyar vera:", p, files)

Patients: 369


In [10]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

MODALITIES = ["flair", "t1", "t1ce", "t2"]

def normalize(vol):
    brain = vol > 0                          # brain voxels mattum (True/False map)
    mean = vol[brain].mean()
    std = vol[brain].std()
    out = np.zeros_like(vol, dtype=np.float32)
    out[brain] = (vol[brain] - mean) / std   # z = (x − μ) ÷ σ
    return out

def load_patient(p):
    folder = f"{root}/{p}"
    scans = []
    for m in MODALITIES:
        vol = nib.load(f"{folder}/{p}_{m}.nii").get_fdata()
        scans.append(normalize(vol))
    image = np.stack(scans, axis=0)          # 4 scans → (4, 240, 240, 155)

    seg = nib.load(f"{folder}/{p}_seg.nii").get_fdata().astype(np.uint8)
    seg[seg == 4] = 3                        # labels: 0,1,2,4 → 0,1,2,3

    image = image[:, 56:184, 56:184, 13:141] # crop
    seg = seg[56:184, 56:184, 13:141]
    return image, seg

image, seg = load_patient("BraTS20_Training_001")

print("Image shape:", image.shape)
print("Seg shape  :", seg.shape)
print("Seg labels :", np.unique(seg))
flair = image[0]
print(f"FLAIR brain mean: {flair[flair != 0].mean():.3f}, std: {flair[flair != 0].std():.3f}")
print("Tumour voxels after crop:", np.sum(seg > 0))

Image shape: (4, 128, 128, 128)
Seg shape  : (128, 128, 128)
Seg labels : [0 1 2 3]
FLAIR brain mean: 0.044, std: 1.035
Tumour voxels after crop: 211033


In [11]:
from tqdm import tqdm

out_dir = "/content/preprocessed"
os.makedirs(out_dir, exist_ok=True)          # folder create (irundhaa paravaayilla)

crop_loss = []

for p in tqdm(patients):
    image, seg = load_patient(p)
    np.save(f"{out_dir}/{p}_image.npy", image.astype(np.float16))
    np.save(f"{out_dir}/{p}_seg.npy", seg)

    # crop evvalavu tumour-ai vettudhu?
    full_seg = nib.load(f"{root}/{p}/{p}_seg.nii").get_fdata()
    full = np.sum(full_seg > 0)
    kept = np.sum(seg > 0)
    crop_loss.append((p, 100 * (full - kept) / full))

losses = np.array([l for _, l in crop_loss])
print(f"\nAverage crop loss: {losses.mean():.3f} %")
print(f"Max crop loss    : {losses.max():.3f} %")
print(f"Patients with > 1% loss: {np.sum(losses > 1)}")

print("\nWorst 5:")
for p, l in sorted(crop_loss, key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {p}: {l:.2f} %")

100%|██████████| 369/369 [10:37<00:00,  1.73s/it]


Average crop loss: 4.104 %
Max crop loss    : 61.411 %
Patients with > 1% loss: 148

Worst 5:
  BraTS20_Training_314: 61.41 %
  BraTS20_Training_277: 48.31 %
  BraTS20_Training_366: 44.35 %
  BraTS20_Training_181: 40.87 %
  BraTS20_Training_217: 38.75 %


In [12]:
!du -sh /content/preprocessed
!ls /content/preprocessed | wc -l

6.5G	/content/preprocessed
738


In [13]:
lo = [56, 56, 13]      # crop start (x, y, z)
hi = [184, 184, 141]   # crop end

mins, maxs = [], []
for p in tqdm(patients):
    s = nib.load(f"{root}/{p}/{p}_seg.nii").get_fdata()
    idx = np.argwhere(s > 0)              # ellaa tumour voxels-oda (x, y, z) positions
    mins.append(idx.min(axis=0))          # tumour-oda chinna x, y, z
    maxs.append(idx.max(axis=0))          # tumour-oda periya x, y, z

mins, maxs = np.array(mins), np.array(maxs)

for a, name in enumerate(["x", "y", "z"]):
    outside = np.sum((mins[:, a] < lo[a]) | (maxs[:, a] >= hi[a]))
    print(f"{name}: tumour range {mins[:, a].min()} – {maxs[:, a].max()} | "
          f"crop {lo[a]}–{hi[a]} | patients outside: {outside}")

100%|██████████| 369/369 [00:28<00:00, 12.72it/s]

x: tumour range 45 – 239 | crop 56–184 | patients outside: 96
y: tumour range 29 – 215 | crop 56–184 | patients outside: 192
z: tumour range 15 – 143 | crop 13–141 | patients outside: 6


In [14]:
lo = [40, 24, 8]
hi = [200, 216, 152]

for a, name in enumerate(["x", "y", "z"]):
    outside = np.sum((mins[:, a] < lo[a]) | (maxs[:, a] >= hi[a]))
    print(f"{name}: crop {lo[a]}–{hi[a]} (size {hi[a]-lo[a]}) | patients outside: {outside}")

# x-la andha 239 yaaru?
print("\nx max top 5:", sorted(maxs[:, 0])[-5:])
print("Patients with x max > 200:", [patients[i] for i in np.where(maxs[:, 0] >= 200)[0]])

x: crop 40–200 (size 160) | patients outside: 2
y: crop 24–216 (size 192) | patients outside: 0
z: crop 8–152 (size 144) | patients outside: 0

x max top 5: [np.int64(191), np.int64(191), np.int64(193), np.int64(212), np.int64(239)]
Patients with x max > 200: ['BraTS20_Training_091', 'BraTS20_Training_344']


In [15]:
for p in ["BraTS20_Training_091", "BraTS20_Training_344"]:
    s = nib.load(f"{root}/{p}/{p}_seg.nii").get_fdata()
    out = np.sum(s[200:, :, :] > 0)      # x = 200 mudhal 239 varaikkum
    total = np.sum(s > 0)
    print(f"{p}: {out} voxels outside of {total} ({100 * out / total:.3f} %)")

BraTS20_Training_091: 4 voxels outside of 149594 (0.003 %)
BraTS20_Training_344: 18 voxels outside of 79098 (0.023 %)


In [16]:
MODALITIES = ["flair", "t1", "t1ce", "t2"]
X0, X1 = 40, 200     # 160
Y0, Y1 = 24, 216     # 192
Z0, Z1 = 8, 152      # 144

def normalize(vol):
    brain = vol > 0
    out = np.zeros_like(vol, dtype=np.float32)
    out[brain] = (vol[brain] - vol[brain].mean()) / vol[brain].std()
    return out

def load_patient(p):
    folder = f"{root}/{p}"
    scans = [normalize(nib.load(f"{folder}/{p}_{m}.nii").get_fdata()) for m in MODALITIES]
    image = np.stack(scans, axis=0)

    seg = nib.load(f"{folder}/{p}_seg.nii").get_fdata().astype(np.uint8)
    seg[seg == 4] = 3
    full = np.sum(seg > 0)                       # crop-ku munnaadi tumour count

    image = image[:, X0:X1, Y0:Y1, Z0:Z1]
    seg = seg[X0:X1, Y0:Y1, Z0:Z1]
    return image, seg, full

out_dir = "/content/preprocessed"
os.makedirs(out_dir, exist_ok=True)

loss = []
for p in tqdm(patients):
    image, seg, full = load_patient(p)
    np.save(f"{out_dir}/{p}_image.npy", image.astype(np.float16))
    np.save(f"{out_dir}/{p}_seg.npy", seg)
    loss.append(100 * (full - np.sum(seg > 0)) / full)

loss = np.array(loss)
print("Shape:", image.shape, seg.shape)
print(f"Avg loss: {loss.mean():.4f} % | Max: {loss.max():.4f} % | >1%: {np.sum(loss > 1)}")
!du -sh {out_dir}

100%|██████████| 369/369 [13:20<00:00,  2.17s/it]

Shape: (4, 160, 192, 144) (160, 192, 144)
Avg loss: 0.0001 % | Max: 0.0228 % | >1%: 0
14G	/content/preprocessed


In [17]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    """3×3×3 conv → norm → activation, rendu thadavai"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.InstanceNorm3d(out_ch),
            nn.LeakyReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNet3D(nn.Module):
    def __init__(self, in_ch=4, n_classes=4, base=16):
        super().__init__()
        c = [base, base*2, base*4, base*8, base*16]      # 16, 32, 64, 128, 256
        self.pool = nn.MaxPool3d(2)                      # size-a paadhiyaakkum

        # Encoder (left)
        self.enc1 = DoubleConv(in_ch, c[0])
        self.enc2 = DoubleConv(c[0], c[1])
        self.enc3 = DoubleConv(c[1], c[2])
        self.enc4 = DoubleConv(c[2], c[3])
        self.bottom = DoubleConv(c[3], c[4])

        # Decoder (right): up = size ×2, dec = skip-oda serthu conv
        self.up4 = nn.ConvTranspose3d(c[4], c[3], 2, stride=2)
        self.dec4 = DoubleConv(c[3] * 2, c[3])
        self.up3 = nn.ConvTranspose3d(c[3], c[2], 2, stride=2)
        self.dec3 = DoubleConv(c[2] * 2, c[2])
        self.up2 = nn.ConvTranspose3d(c[2], c[1], 2, stride=2)
        self.dec2 = DoubleConv(c[1] * 2, c[1])
        self.up1 = nn.ConvTranspose3d(c[1], c[0], 2, stride=2)
        self.dec1 = DoubleConv(c[0] * 2, c[0])

        self.out = nn.Conv3d(c[0], n_classes, 1)         # 16 channels → 4 class scores

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottom(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))   # skip connection!
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)

# ---------- Test ----------
device = "cuda"
model = UNet3D().to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")

first = model.enc1.block[0]
print("First conv params:", first.weight.numel() + first.bias.numel())

x = torch.randn(1, 4, 128, 128, 128, device=device)   # oru fake 128³ patch
with torch.no_grad():
    y = model(x)
print("Input :", tuple(x.shape))
print("Output:", tuple(y.shape))
print(f"GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Total parameters: 5,646,260
First conv params: 1744
Input : (1, 4, 128, 128, 128)
Output: (1, 4, 128, 128, 128)
GPU memory: 0.85 GB


In [18]:
import random
from torch.utils.data import Dataset, DataLoader

# ---------- 1. Split ----------
random.seed(42)
shuffled = patients.copy()
random.shuffle(shuffled)

n_train = int(0.70 * len(shuffled))
n_val   = int(0.15 * len(shuffled))
train_ids = shuffled[:n_train]
val_ids   = shuffled[n_train:n_train + n_val]
test_ids  = shuffled[n_train + n_val:]
print("Train / Val / Test:", len(train_ids), len(val_ids), len(test_ids))

# ---------- 2. Random patch ----------
PATCH = 128

def random_patch(image, seg, size=PATCH, tumour_prob=0.7):
    _, X, Y, Z = image.shape                         # 4, 160, 192, 144
    if random.random() < tumour_prob:
        idx = np.argwhere(seg > 0)                   # ellaa tumour voxels
        cx, cy, cz = idx[random.randrange(len(idx))] # oru tumour voxel random-aa
        x0 = int(np.clip(cx - size // 2, 0, X - size))   # adhai naduvula vai,
        y0 = int(np.clip(cy - size // 2, 0, Y - size))   # aana box-ku veliya
        z0 = int(np.clip(cz - size // 2, 0, Z - size))   # pogaama clip pannu
    else:
        x0 = random.randint(0, X - size)
        y0 = random.randint(0, Y - size)
        z0 = random.randint(0, Z - size)
    return (image[:, x0:x0+size, y0:y0+size, z0:z0+size],
            seg[x0:x0+size, y0:y0+size, z0:z0+size])

# ---------- 3. Dataset ----------
class BraTSDataset(Dataset):
    def __init__(self, ids, train=True):
        self.ids = ids
        self.train = train

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        p = self.ids[i]
        image = np.load(f"{out_dir}/{p}_image.npy").astype(np.float32)
        seg = np.load(f"{out_dir}/{p}_seg.npy").astype(np.int64)
        if self.train:
            image, seg = random_patch(image, seg)    # train: 128³ patch
        return torch.from_numpy(image.copy()), torch.from_numpy(seg.copy())
                                                     # val/test: muzhu 160×192×144

train_ds = BraTSDataset(train_ids, train=True)
val_ds   = BraTSDataset(val_ids, train=False)
train_dl = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2)
val_dl   = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

# ---------- 4. Check ----------
x, y = next(iter(train_dl))
print("Train batch:", tuple(x.shape), x.dtype, "|", tuple(y.shape), y.dtype)
x, y = next(iter(val_dl))
print("Val batch  :", tuple(x.shape), "|", tuple(y.shape))

fracs = [(train_ds[i][1] > 0).float().mean().item() * 100 for i in range(30)]
print(f"Avg tumour % in 30 training patches: {np.mean(fracs):.2f} %")

Train / Val / Test: 258 55 56
Train batch: (1, 4, 128, 128, 128) torch.float32 | (1, 128, 128, 128) torch.int64
Val batch  : (1, 4, 160, 192, 144) | (1, 160, 192, 144)
Avg tumour % in 30 training patches: 5.11 %


In [19]:
import torch.nn.functional as F
import time

def dice_loss(logits, target, eps=1e-5):
    probs = torch.softmax(logits, dim=1)                          # scores → probabilities
    onehot = F.one_hot(target, 4).permute(0, 4, 1, 2, 3).float()  # label 2 → [0,0,1,0]
    dims = (0, 2, 3, 4)                                           # ellaa voxels-um kootti
    inter = (probs * onehot).sum(dims)                            # Σ p·g  (per class)
    total = probs.sum(dims) + onehot.sum(dims)                    # Σp + Σg
    dice = (2 * inter + eps) / (total + eps)
    return 1 - dice[1:].mean()                                    # background (class 0) skip

ce = nn.CrossEntropyLoss()
def loss_fn(logits, target):
    return ce(logits, target) + dice_loss(logits, target)

model = UNet3D().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
scaler = torch.amp.GradScaler("cuda")

torch.cuda.reset_peak_memory_stats()
model.train()
it = iter(train_dl)

for step in range(10):
    x, y = next(it)
    x, y = x.to(device), y.to(device)
    t0 = time.time()

    optimizer.zero_grad()                              # pazhaya gradients clear
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(x)                              # 1. forward: model badhil
    loss = loss_fn(logits.float(), y)                  # 2. evvalavu thappu
    scaler.scale(loss).backward()                      # 3. saivu (gradient) kanakku
    scaler.step(optimizer)                             # 4. weights-a konjam maatthu
    scaler.update()

    torch.cuda.synchronize()
    print(f"step {step}: loss {loss.item():.4f} | {time.time() - t0:.2f} s")

print(f"\nPeak GPU memory (training): {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

step 0: loss 2.4082 | 1.26 s
step 1: loss 2.4081 | 0.21 s
step 2: loss 2.3719 | 0.21 s
step 3: loss 2.2667 | 0.21 s
step 4: loss 2.3498 | 0.21 s
step 5: loss 2.1845 | 0.21 s
step 6: loss 2.2678 | 0.21 s
step 7: loss 2.2206 | 0.21 s
step 8: loss 2.0818 | 0.21 s
step 9: loss 2.0951 | 0.21 s

Peak GPU memory (training): 1.38 GB
